<a href="https://colab.research.google.com/github/Ikbal-ullah/Algo-trading/blob/main/day1%2B2_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import pandas as pd
from datetime import datetime

# 1. Define target coordinates for Assam (Guwahati)
LATITUDE = 26.14
LONGITUDE = 91.73

# 2. Construct API URL targeting specific parameters critical to Culex vectors
url = f"[https://api.open-meteo.com/v1/forecast?latitude=](https://api.open-meteo.com/v1/forecast?latitude=){LATITUDE}&longitude={LONGITUDE}&current=temperature_2m,relative_humidity_2m,rain&timezone=auto"

print("Fetching live weather data from Assam...")
response = requests.get(url)

if response.status_code == 200:
    data = response.json()
    current_weather = data['current']

    # 3. Restructure raw JSON payload into a clean data dictionary
    weather_report = {
        "Timestamp": [datetime.now().strftime("%Y-%m-%d %H:%M:%S")],
        "Location": ["Assam (Guwahati)"],
        "Temperature (°C)": [current_weather['temperature_2m']],
        "Humidity (%)": [current_weather['relative_humidity_2m']],
        "Rainfall (mm)": [current_weather['rain']]
    }

    # 4. Initialize a Pandas DataFrame (the fundamental matrix structure for ML)
    df = pd.DataFrame(weather_report)
    print("\\n--- LIVE DATA SECURED ---")
    print(df)
else:
    print(f"Failed to fetch data. Error code: {response.status_code}")

In [ ]:
import requests
import pandas as pd

LATITUDE = 26.14
LONGITUDE = 91.73

# Modified query parameter adding 'past_days=30' to look backward in time
url = f"[https://api.open-meteo.com/v1/forecast?latitude=](https://api.open-meteo.com/v1/forecast?latitude=){LATITUDE}&longitude={LONGITUDE}&daily=temperature_2m_max,relative_humidity_2m_max,precipitation_sum&past_days=30&forecast_days=0&timezone=auto"

print("Extracting 30-day historical climate trend for Assam...")
response = requests.get(url)

if response.status_code == 200:
    data = response.json()
    daily_data = data['daily']

    # Structural mapping of time-series lists
    history_report = {
        "Date": daily_data['time'],
        "Max Temp (°C)": daily_data['temperature_2m_max'],
        "Max Humidity (%)": daily_data['relative_humidity_2m_max'],
        "Total Rain (mm)": daily_data['precipitation_sum']
    }

    df_history = pd.DataFrame(history_report)
    print("\\n--- 30-DAY HISTORICAL DATASET SECURED ---")
    print(df_history.head(10))
else:
    print(f"Failed to fetch historical data. Error: {response.status_code}")

In [ ]:
# Create a engineered feature: 3-Day Rolling Window Accumulation
df_history['3_Day_Accumulated_Rain (mm)'] = df_history['Total Rain (mm)'].rolling(window=3).sum()

print("--- FEATURE ENGINEERING COMPLETE ---")
print(df_history.head(10))

In [1]:
import requests
import pandas as pd
import time
from datetime import datetime, timedelta

# 1. Define coordinate matrices for major district hubs across Assam's zones
DISTRICTS = {
    "Kamrup_Metropolitan": {"lat": 26.11, "lon": 91.74},  # Lower Assam / Guwahati
    "Dibrugarh": {"lat": 27.48, "lon": 94.90},            # Upper Assam (High Risk)
    "Cachar": {"lat": 24.81, "lon": 92.80},               # Barak Valley
    "Nagaon": {"lat": 26.35, "lon": 92.68},               # Central Assam
    "Sonitpur": {"lat": 26.63, "lon": 92.79},             # North Assam
    "Tinsukia": {"lat": 27.50, "lon": 95.36},             # Far Upper Assam
    "Dhubri": {"lat": 26.03, "lon": 89.97}                # West Border Zone
}

# 2. Dynamic temporal range definition (Past 5 years up to last week)
end_date = (datetime.now() - timedelta(days=5)).strftime("%Y-%m-%d")
start_date = (datetime.now() - timedelta(days=5*365)).strftime("%Y-%m-%d")

print(f"Target Time Horizon: From {start_date} to {end_date}")
all_district_data = []

# 3. Begin the global iteration loop
for district_name, coords in DISTRICTS.items():
    print(f"\n📡 Querying historical climate matrix for: {district_name}...")

    # Using Open-Meteo Historical Archive API endpoint
    archive_url = (
        f"https://archive-api.open-meteo.com/v1/archive?"
        f"latitude={coords['lat']}&longitude={coords['lon']}&"
        f"start_date={start_date}&end_date={end_date}&"
        f"daily=temperature_2m_max,relative_humidity_2m_max,precipitation_sum&"
        f"timezone=auto"
    )

    response = requests.get(archive_url)

    if response.status_code == 200:
        raw_payload = response.json()
        daily_records = raw_payload['daily']

        # Structure payload matching this specific iteration's spatial context
        temp_df = pd.DataFrame({
            "Date": daily_records['time'],
            "District": district_name,
            "Latitude": coords['lat'],
            "Longitude": coords['lon'],
            "Max_Temp_C": daily_records['temperature_2m_max'],
            "Max_Humidity_Pct": daily_records['relative_humidity_2m_max'],
            "Daily_Rain_mm": daily_records['precipitation_sum']
        })

        all_district_data.append(temp_df)
        print(f"✅ Secured {len(temp_df)} daily data points for {district_name}.")

        # Defensive API Programming: Explicit 1-second rest to respect free-tier endpoints
        time.sleep(1)
    else:
        print(f"❌ Failed to extract data for {district_name}. HTTP Error: {response.status_code}")

# 4. Consolidate individual matrices into our foundational dataframe
if all_district_data:
    assam_historical_raw = pd.concat(all_district_data, ignore_index=True)
    print("\n🏁 LOOPING PIPELINE OPERATION COMPLETE 🏁")
    print(f"Total Structural Dataset Shape: {assam_historical_raw.shape}")
    print("\nSample Preview of the Master Frame:")
    print(assam_historical_raw.sample(5)) # Pulls 5 random rows across space and time
else:
    print("Zero datasets compiled.")

Target Time Horizon: From 2021-05-26 to 2026-05-20

📡 Querying historical climate matrix for: Kamrup_Metropolitan...
✅ Secured 1821 daily data points for Kamrup_Metropolitan.

📡 Querying historical climate matrix for: Dibrugarh...
✅ Secured 1821 daily data points for Dibrugarh.

📡 Querying historical climate matrix for: Cachar...
✅ Secured 1821 daily data points for Cachar.

📡 Querying historical climate matrix for: Nagaon...
✅ Secured 1821 daily data points for Nagaon.

📡 Querying historical climate matrix for: Sonitpur...
✅ Secured 1821 daily data points for Sonitpur.

📡 Querying historical climate matrix for: Tinsukia...
✅ Secured 1821 daily data points for Tinsukia.

📡 Querying historical climate matrix for: Dhubri...
✅ Secured 1821 daily data points for Dhubri.

🏁 LOOPING PIPELINE OPERATION COMPLETE 🏁
Total Structural Dataset Shape: (12747, 7)

Sample Preview of the Master Frame:
             Date   District  Latitude  Longitude  Max_Temp_C  \
10616  2025-07-15   Tinsukia     27.5

In [2]:
# This takes our master table and saves it as a permanent Excel/CSV file inside Colab
assam_historical_raw.to_csv('assam_historical_weather.csv', index=False)
print("🎯 SUCCESS: Your 12,747-row dataset is now saved as 'assam_historical_weather.csv'!")

🎯 SUCCESS: Your 12,747-row dataset is now saved as 'assam_historical_weather.csv'!


In [3]:
import requests
import pandas as pd
import time
from datetime import datetime, timedelta

# 1. Target your home district coordinates
NALBARI = {"lat": 26.45, "lon": 91.44}
district_name = "Nalbari"

# 2. Maintain the exact same temporal horizon (Past 5 years)
end_date = (datetime.now() - timedelta(days=5)).strftime("%Y-%m-%d")
start_date = (datetime.now() - timedelta(days=5*365)).strftime("%Y-%m-%d")

print(f"Targeting Nalbari Timeline: From {start_date} to {end_date}")

# 3. Requesting the Open-Meteo Archive
archive_url = (
    f"https://archive-api.open-meteo.com/v1/archive?"
    f"latitude={NALBARI['lat']}&longitude={NALBARI['lon']}&"
    f"start_date={start_date}&end_date={end_date}&"
    f"daily=temperature_2m_max,relative_humidity_2m_max,precipitation_sum&"
    f"timezone=auto"
)

response = requests.get(archive_url)

if response.status_code == 200:
    daily_records = response.json()['daily']

    # Structure the Nalbari matrix
    df_nalbari = pd.DataFrame({
        "Date": daily_records['time'],
        "District": district_name,
        "Latitude": NALBARI['lat'],
        "Longitude": NALBARI['lon'],
        "Max_Temp_C": daily_records['temperature_2m_max'],
        "Max_Humidity_Pct": daily_records['relative_humidity_2m_max'],
        "Daily_Rain_mm": daily_records['precipitation_sum']
    })
    print(f"✅ Secured {len(df_nalbari)} daily data points for {district_name}!")

    # 4. Read the existing master dataset and append Nalbari seamlessly
    try:
        # Check if the previous master exists in memory, or reload it
        if 'assam_historical_raw' in locals():
            assam_historical_raw = pd.concat([assam_historical_raw, df_nalbari], ignore_index=True)
        else:
            print("Reading your existing backup dataset file to patch...")
            master_disk = pd.read_csv('assam_historical_weather.csv')
            assam_historical_raw = pd.concat([master_disk, df_nalbari], ignore_index=True)

        # Save the updated master back to file
        assam_historical_raw.to_csv('assam_historical_weather.csv', index=False)
        print("\n🏁 NALBARI DIRECTORY INTERACTION COMPLETE 🏁")
        print(f"Updated Master Dataset Total Shape: {assam_historical_raw.shape}")
        print("\nVerification - Checking Nalbari data slice sample:")
        print(assam_historical_raw[assam_historical_raw['District'] == 'Nalbari'].tail(5))

    except Exception as e:
        print(f"Data mapping error: {e}. Let's write the Nalbari slice to a separate file for safety.")
        df_nalbari.to_csv('nalbari_historical_weather.csv', index=False)
else:
    print(f"API rejection error code: {response.status_code}")

Targeting Nalbari Timeline: From 2021-05-26 to 2026-05-20
✅ Secured 1821 daily data points for Nalbari!

🏁 NALBARI DIRECTORY INTERACTION COMPLETE 🏁
Updated Master Dataset Total Shape: (14568, 7)

Verification - Checking Nalbari data slice sample:
             Date District  Latitude  Longitude  Max_Temp_C  Max_Humidity_Pct  \
14563  2026-05-16  Nalbari     26.45      91.44        29.5                98   
14564  2026-05-17  Nalbari     26.45      91.44        29.6                99   
14565  2026-05-18  Nalbari     26.45      91.44        28.3                99   
14566  2026-05-19  Nalbari     26.45      91.44        31.3                98   
14567  2026-05-20  Nalbari     26.45      91.44        32.5                94   

       Daily_Rain_mm  
14563           27.4  
14564           87.8  
14565           29.4  
14566            1.4  
14567            3.4  


In [6]:
import requests
import pandas as pd
import time
from datetime import datetime, timedelta
# 4. APPEND ENGINE: Read your existing list and attach Nalbari to the bottom
try:
        # Check if the earlier 7-district list is active in Colab memory
    if 'assam_historical_raw' in locals() and not assam_historical_raw.empty:
      print("🔄 Memory check passed. Appending Nalbari to the existing variable...")
            # Filter out any old Nalbari rows if you accidentally ran it before to prevent duplicates
      clean_master = assam_historical_raw[assam_historical_raw['District'] != 'Nalbari']
      assam_historical_raw = pd.concat([clean_master, df_nalbari], ignore_index=True)
    else:
            # If Colab restarted, read the file we saved yesterday and append to it
      print("📂 Reloading 'assam_historical_weather.csv' from disk to patch...")
      master_disk = pd.read_csv('assam_historical_weather.csv')
      clean_master = master_disk[master_disk['District'] != 'Nalbari']
      assam_historical_raw = pd.concat([clean_master, df_nalbari], ignore_index=True)

        # 5. SAVE: Save the final unified dataset back as a permanent CSV
    assam_historical_raw.to_csv('assam_historical_weather.csv', index=False)

    print("\n🏁 CONCATENATION SUCCESSFUL 🏁")
    print(f"📈 Total Rows in Matrix (All 8 Districts): {assam_historical_raw.shape[0]}")
    print("\n👇 Here is a live look at your updated dataset with Nalbari verified at the end:")
    print(assam_historical_raw[assam_historical_raw['District'] == 'Nalbari'].tail(5))

except Exception as e:
    print(f"⚠️ Could not merge automatically because: {e}")
    print("Saving Nalbari separately as a backup file: 'nalbari_historical_weather.csv'")
    df_nalbari.to_csv('nalbari_historical_weather.csv', index=False)

🔄 Memory check passed. Appending Nalbari to the existing variable...

🏁 CONCATENATION SUCCESSFUL 🏁
📈 Total Rows in Matrix (All 8 Districts): 14568

👇 Here is a live look at your updated dataset with Nalbari verified at the end:
             Date District  Latitude  Longitude  Max_Temp_C  Max_Humidity_Pct  \
14563  2026-05-16  Nalbari     26.45      91.44        29.5                98   
14564  2026-05-17  Nalbari     26.45      91.44        29.6                99   
14565  2026-05-18  Nalbari     26.45      91.44        28.3                99   
14566  2026-05-19  Nalbari     26.45      91.44        31.3                98   
14567  2026-05-20  Nalbari     26.45      91.44        32.5                94   

       Daily_Rain_mm  
14563           27.4  
14564           87.8  
14565           29.4  
14566            1.4  
14567            3.4  
